# Penyelarasan Data Temporal — Berita Geopolitik & Kurs USD/IDR

Notebook ini menyelaraskan data berita geopolitik (`geopolitical_news.csv`, hasil `scraper.ipynb`) dengan data kurs harian USD/IDR dari Bank Indonesia (`jisdor_usd_idr.csv`, hasil `timeseries_jisdor.ipynb`) berdasarkan waktu terbit beritanya.

## 0. Import Library

In [20]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 80)

## 1. Latar Belakang Masalah

Dua sumber data yang kita punya punya karakteristik waktu yang berbeda:

- **Data kurs (JISDOR)** hanya tersedia di hari kerja pasar sehingga tidak ada nilai di akhir pekan atau hari libur nasional.
- **Data berita** bisa terbit kapan saja, 24 jam, termasuk akhir pekan dan hari libur.

Akibatnya, sebuah berita yang terbit di hari Sabtu, Minggu, hari libur, atau setelah jam tutup pasar pada hari kerja, tidak punya "pasangan" langsung di data kurs pada tanggal yang sama persis. Notebook ini menetapkan aturan logis untuk memetakan tiap berita ke hari trading yang paling relevan, tanpa membuang informasinya.

## Load Data

In [21]:
DATA_DIR = Path("../data")

kurs = pd.read_csv(DATA_DIR / "processed" / "jisdor_usd_idr.csv", parse_dates=["date"])
kurs = kurs.sort_values("date").reset_index(drop=True)

news = pd.read_csv(DATA_DIR / "processed" / "articles_clean.csv", parse_dates=["published_at"])  # sudah dibersihkan preprocessing.ipynb

print(f"Data kurs   : {len(kurs)} baris, {kurs['date'].min().date()} s.d. {kurs['date'].max().date()}")
print(f"Data berita : {len(news)} baris, {news['published_at'].min()} s.d. {news['published_at'].max()}")

kurs.head()

Data kurs   : 1202 baris, 2021-09-01 s.d. 2026-09-01
Data berita : 14411 baris, 2021-09-01 07:00:00 s.d. 2026-08-31 07:00:00


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


In [22]:
news.head()

,published_at,date,clean_title,lead_paragraph,clean_content,dateline_location,word_count,sentence_count,paragraph_count,is_length_outlier,source_domain,language,url
0,2021-09-01 07:00:00,2021-09-01,"I’ve paid off almost $200,000 of student loans in 3 years: Here's my best ad...","In 2015, I graduated law school with six figures of student loan debt. Since...","In 2015, I graduated law school with six figures of student loan debt. Since...",NaN,1396,61,43,False,CNBC,en,https://www.cnbc.com/2021/09/01/i-paid-off-almost-200k-of-student-loans-in-a...
1,2021-09-01 07:00:00,2021-09-01,Weekly mortgage-refinance demand drops as interest rates stall,A prolonged period of low mortgage rates is taking its toll on the refinance...,A prolonged period of low mortgage rates is taking its toll on the refinance...,NaN,323,12,8,False,CNBC,en,https://www.cnbc.com/2021/09/01/weekly-mortgage-refinance-demand-drops-as-in...
2,2021-09-01 07:00:00,2021-09-01,S&P 500 and Nasdaq notch record closes as jobless claims reach pandemic low,The S&P 500 and the Nasdaq Composite climbed to new respective records on Th...,The S&P 500 and the Nasdaq Composite climbed to new respective records on Th...,NaN,394,17,9,False,CNBC,en,https://www.cnbc.com/2021/09/01/stock-market-futures-open-to-close-news.html
3,2021-09-01 07:00:00,2021-09-01,"U.S. relationship with Taliban unclear after end of Afghanistan War, senior ...",Secretary of Defense Lloyd Austin said Wednesday it was not yet clear what k...,Secretary of Defense Lloyd Austin said Wednesday it was not yet clear what k...,WASHINGTON,675,37,17,False,CNBC,en,https://www.cnbc.com/2021/09/01/afghanistan-update-us-relationship-with-tali...
4,2021-09-01 07:00:00,2021-09-01,"China's health-care sector could be Beijing's next regulatory target, analys...","China's health-care sector will probably be the next to fall under scrutiny,...","China's health-care sector will probably be the next to fall under scrutiny,...",NaN,515,24,20,False,CNBC,en,https://www.cnbc.com/2021/09/02/chinas-regulatory-crackdown-on-healthcare-se...


## 3. Konversi Zona Waktu

Kolom `published_at` berasal dari `pubDate` RSS Google News, yang formatnya GMT (=UTC). Data kurs JISDOR mengikuti jam operasional pasar Indonesia (WIB, UTC+7). Supaya aturan "jam berapa market tutup" bisa diterapkan dengan benar, kedua data ini perlu disamakan zona waktunya dulu, kita konversi timestamp berita dari UTC ke WIB.

In [23]:
news["published_utc"] = news["published_at"].dt.tz_localize("UTC")
news["published_wib"] = news["published_utc"].dt.tz_convert("Asia/Jakarta")

news[["published_at", "published_wib"]].head()

,published_at,published_wib
0,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
1,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
2,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
3,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
4,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00


## 4. Aturan Penyelarasan Temporal

Aturan yang digunakan (roll-forward ke hari trading berikutnya):

1. Daftar "hari trading valid" diambil langsung dari tanggal-tanggal yang ada di data kurs JISDOR, jadi hari libur nasional maupun akhir pekan otomatis terdeteksi tanpa perlu kalender libur terpisah, karena BI memang tidak mempublikasikan kurs pada hari-hari tersebut.
2. Ditetapkan jam cutoff 16:00 WIB sebagai asumsi jam tutup pasar valas domestik. Berita yang terbit sebelum jam ini pada hari kerja dianggap mempengaruhi kurs hari itu juga.
3. Berita yang terbit **setelah** jam cutoff, atau pada akhir pekan/hari libur, digeser maju ke hari trading valid berikutnya.
4. Berita tidak dihapus, hanya dipetakan ulang ke tanggal yang paling masuk akal secara logis, supaya sinyalnya tetap terpakai untuk pengujian hipotesis di tahap selanjutnya.

In [24]:
CUTOFF_JAM = 16  # asumsi: pasar valas domestik dianggap tutup jam 16:00 WIB

trading_days = pd.DatetimeIndex(kurs["date"].dt.normalize().unique())


def geser_ke_trading_day(ts, trading_days=trading_days, cutoff=CUTOFF_JAM):
    """Memetakan satu timestamp berita ke hari trading valid berikutnya."""
    tanggal = ts.normalize().tz_localize(None)
    if ts.hour >= cutoff:
        tanggal += pd.Timedelta(days=1)
    while tanggal not in trading_days:
        tanggal += pd.Timedelta(days=1)
        if tanggal > trading_days.max() + pd.Timedelta(days=10):
            return pd.NaT  # di luar cakupan data kurs
    return tanggal


news["trading_day"] = news["published_wib"].apply(geser_ke_trading_day)

n_gagal = news["trading_day"].isna().sum()
print(f"Total berita         : {len(news)}")
print(f"Berhasil dipetakan   : {len(news) - n_gagal}")
print(f"Gagal dipetakan      : {n_gagal} (di luar rentang tanggal data kurs)")

Total berita         : 14411
Berhasil dipetakan   : 14411
Gagal dipetakan      : 0 (di luar rentang tanggal data kurs)


## 5. Agregasi Berita per Hari Trading

Karena satu hari trading bisa menampung lebih dari satu berita (dan sebaliknya, satu hari trading bisa tidak punya berita sama sekali karena strategi *sampling* bi-weekly pada tahap scraping), berita perlu diagregasi dulu menjadi satu baris per hari sebelum digabung ke data kurs. Di sini kita hitung jumlah berita, menggabungkan judul-judulnya, dan menggabungkan isi teks bersihnya (clean_content) per hari sehingga hasil akhir bisa langsung dipakai untuk ekstraksi fitur teks di Tugas 2 tanpa perlu join ulang ke articles_clean.csv

In [25]:
def cap_daily_text(text_series, max_words=1500):
    combined = " | ".join(text_series.dropna())
    words = combined.split()
    return " | ".join(words[:max_words]) if len(words) > max_words else combined

agg = news.dropna(subset=["trading_day"]).groupby("trading_day").agg(
    jumlah_berita=("url", "count"),
    judul_gabungan=("clean_title", lambda x: " | ".join(x)),
    lead_paragraph_gabungan=("lead_paragraph", lambda x: cap_daily_text(x, max_words=1500)),
    konten_gabungan=("clean_content", lambda x: "\n\n---\n\n".join(x)),
).reset_index()

agg.head()

,trading_day,jumlah_berita,judul_gabungan,lead_paragraph_gabungan,konten_gabungan
0,2021-09-01,5,"I’ve paid off almost $200,000 of student loans in 3 years: Here's my best ad...","In 2015, I graduated law school with six figures of student loan debt. Since...","In 2015, I graduated law school with six figures of student loan debt. Since..."
1,2021-09-02,5,Supreme Court refuses to block Texas law that bans most abortions | If you c...,The Supreme Court declined to block a Texas law banning most abortions in a ...,The Supreme Court declined to block a Texas law banning most abortions in a ...
2,2021-09-03,4,"Jobs report disappoints — only 235,000 positions added vs. expectations of 7...","Job creation for August was a huge disappointment, with the economy adding j...","Job creation for August was a huge disappointment, with the economy adding j..."
3,2021-09-06,7,"'Stagflation' is the greatest threat to Europe’s recovery, warns ex-Italian ...",Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...
4,2021-09-07,6,"Malaysia will start treating Covid as 'endemic' around end-October, trade mi...",Malaysia will start treating Covid-19 as an endemic disease around the end o...,Malaysia will start treating Covid-19 as an endemic disease around the end o...


## 6. Penggabungan dengan Data Kurs 

In [26]:
final = kurs.merge(agg, left_on="date", right_on="trading_day", how="left")
final["jumlah_berita"] = final["jumlah_berita"].fillna(0).astype(int)
final["judul_gabungan"] = final["judul_gabungan"].fillna("")
final["lead_paragraph_gabungan"] = final["lead_paragraph_gabungan"].fillna("")
final["konten_gabungan"] = final["konten_gabungan"].fillna("")
final = final.drop(columns=["trading_day"])

final.head(10)

,date,usd_idr,jumlah_berita,judul_gabungan,lead_paragraph_gabungan,konten_gabungan
0,2021-09-01,14284,5,"I’ve paid off almost $200,000 of student loans in 3 years: Here's my best ad...","In 2015, I graduated law school with six figures of student loan debt. Since...","In 2015, I graduated law school with six figures of student loan debt. Since..."
1,2021-09-02,14281,5,Supreme Court refuses to block Texas law that bans most abortions | If you c...,The Supreme Court declined to block a Texas law banning most abortions in a ...,The Supreme Court declined to block a Texas law banning most abortions in a ...
2,2021-09-03,14261,4,"Jobs report disappoints — only 235,000 positions added vs. expectations of 7...","Job creation for August was a huge disappointment, with the economy adding j...","Job creation for August was a huge disappointment, with the economy adding j..."
3,2021-09-06,14239,7,"'Stagflation' is the greatest threat to Europe’s recovery, warns ex-Italian ...",Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...
4,2021-09-07,14195,6,"Malaysia will start treating Covid as 'endemic' around end-October, trade mi...",Malaysia will start treating Covid-19 as an endemic disease around the end o...,Malaysia will start treating Covid-19 as an endemic disease around the end o...
5,2021-09-08,14266,6,BlackRock responds to George Soros' criticism over China investments | Job o...,"BlackRock, the world's largest asset manager, has responded to sharp critici...","BlackRock, the world's largest asset manager, has responded to sharp critici..."
6,2021-09-09,14272,11,China's rumored ambitions to dive into Afghanistan are overstated and unreal...,One of the first things many Western pundits predicted as the chaotic Americ...,One of the first things many Western pundits predicted as the chaotic Americ...
7,2021-09-10,14225,8,Rudy Giuliani associate Igor Fruman pleads guilty to soliciting foreign camp...,"Igor Fruman, a former associate of embattled lawyer Rudy Giuliani, pleaded g...","Igor Fruman, a former associate of embattled lawyer Rudy Giuliani, pleaded g..."
8,2021-09-13,14260,15,California's battle with climate change is at stake in Tuesday's recall elec...,California voters will decide whether to remove Democratic Gov. Gavin Newsom...,California voters will decide whether to remove Democratic Gov. Gavin Newsom...
9,2021-09-14,14257,7,How to track subscriptions you autopay with your credit card | Chinese stude...,Terms apply to American Express benefits and offers. Visit americanexpress.c...,Terms apply to American Express benefits and offers. Visit americanexpress.c...


## 7. Validasi & Ringkasan Hasil

In [27]:
total_hari = len(final)
hari_ada_berita = (final["jumlah_berita"] > 0).sum()

print(f"Total hari trading                : {total_hari}")
print(f"Hari trading dengan >=1 berita     : {hari_ada_berita} ({hari_ada_berita/total_hari:.1%})")
print(f"Hari trading tanpa berita          : {total_hari - hari_ada_berita}")
print(f"Total berita ter-assign            : {final['jumlah_berita'].sum()}")
print(f"Berita di luar cakupan data kurs   : {n_gagal} dari {len(news)} ({n_gagal/len(news):.1%})")
print()

Total hari trading                : 1202
Hari trading dengan >=1 berita     : 1185 (98.6%)
Hari trading tanpa berita          : 17
Total berita ter-assign            : 14411
Berita di luar cakupan data kurs   : 0 dari 14411 (0.0%)



## 8. Menyimpan Hasil Akhir 

In [28]:
OUTPUT_PATH = DATA_DIR / "processed" / "aligned_dataset.csv"
final.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan di: {OUTPUT_PATH.resolve()}")
print(f"Bentuk akhir: {final.shape[0]} baris x {final.shape[1]} kolom")
final.dtypes

Tersimpan di: C:\UGM Stuff\Semester 5\Natural Language Processing\nlp-geopolitics-usa\data\processed\aligned_dataset.csv
Bentuk akhir: 1202 baris x 6 kolom


date                       datetime64[ns]
usd_idr                             int64
jumlah_berita                       int64
judul_gabungan                     object
lead_paragraph_gabungan            object
konten_gabungan                    object
dtype: object